# 🧬 Práctica C: Ensamblaje de Genomas con Python + conda en Google Colab



> [!IMPORTANT]
> **Antes de continuar**, lea la [guía de prácticas compartida](00_genome_assembly_common.md) — contiene el contexto biológico, los tres casos de estudio y las instrucciones para descargar los datos en Colab.

Esta práctica sigue el mismo **flujo de trabajo** que las Prácticas A y B, pero en lugar de Galaxy usa **Google Colab** como plataforma de cómputo y **Python** y **Conda** para instalar las herramientas bioinformáticas necesarias. Esto le permite:

- Ejecutar herramientas de línea de comandos (`fastp`, `SPAdes`, `QUAST`) desde un notebook en la nube.
- Analizar y visualizar los resultados con Python (`pandas`, `matplotlib`).
- Entender cómo se encadenan estas herramientas en un pipeline real.

## ⚙️ Paso 0 — Configurar el entorno con conda

Ejecute esta celda **primero**. Instala Miniconda y todas las herramientas necesarias.
⏱️ Puede tardar 8–12 minutos.

In [ ]:
# Instalamos biopython
!pip install biopython

# Instalamos Conda para colab
!pip install condacolab

import condacolab
condacolab.install()

# Verify condacolab installation
import os
os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']
!conda --version

Una vez que Conda esté instalado, es posible que necesites reiniciar el entorno de ejecución de Colab (Runtime > Restart runtime) para que los cambios surtan efecto. Después de reiniciar, puedes ejecutar comandos `conda` en nuevas celdas de código.

### Instalar herramientas

Ya que hemos instalado **Conda** es necesario instalar las herramientas que vamos a usar para el ensamblaje.

In [ ]:
# Instalar herramientas bioinformáticas

!conda install -c \
  bioconda \
  fastqc \
  trimmomatic \
  bwa \
  samtools \
  bcftools \
  fastp \
  spades \
  quast
!echo "✅ Instalación completa"

ahora vamos a instalar otras herramientas propias de **Python** que nos ayudaran a procesar y ver los resutlados mas facil para los humanos

In [ ]:
!pip install matplotlib pandas
!echo "✅ Instalación completa"

## 📥 Paso 1 — Descargar los datos del caso asignado

Solo trabajaremos con un solo caso, por lo cual debe definir primero el caso que le indicó el profesor (A, B o C). Asigne a la variable el caso que va a trabajar.

Consulte la [guía compartida](00_genome_assembly_common.md#-casos-de-estudio) para el contexto biológico de cada caso.

In [ ]:
# ⚠️ **IMPORTANTE**: Define el caso a trabajar aquí y solo aquí.
# Descomenta la línea que corresponda a tu caso (A, B o C).
CASO = "caso_A" # Por ejemplo, para caso A

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

# Crear el directorio de datos para el caso seleccionado
!mkdir -p {CASO}/data

DOWNLOAD_SUCCESS = False

if CASO == "caso_A":
  print("── CASO A: Staphylococcus aureus MRSA (~2.8 Mb, GC ~33%) ──────────────────")
  !wget -q ftp://ftp.sra.ebi.ac.uk/vol1/fastq/DRR187/DRR187559/DRR187559_1.fastq.gz -O {CASO}/data/DRR187559_1.fastq.gz
  !wget -q ftp://ftp.sra.ebi.ac.uk/vol1/fastq/DRR187/DRR187559/DRR187559_2.fastq.gz -O {CASO}/data/DRR187559_2.fastq.gz
  !wget -q "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/013/425/GCF_000013425.1_ASM1342v1/GCF_000013425.1_ASM1342v1_genomic.fna.gz" -O {CASO}/data/ref.fna.gz
  DOWNLOAD_SUCCESS = True
elif CASO == "caso_B":
  print("── CASO B: Klebsiella pneumoniae (~5.5 Mb, GC ~57%) ───")
  !wget -q ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR148/071/ERR14828471/ERR14828471_1.fastq.gz -O {CASO}/data/ERR14828471_1.fastq.gz
  !wget -q ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR148/071/ERR14828471/ERR14828471_2.fastq.gz -O {CASO}/data/ERR14828471_2.fastq.gz
  !wget -q "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/240/185/GCF_000240185.1_ASM24018v2/GCF_000240185.1_ASM24018v2_genomic.fna.gz" -O {CASO}/data/ref.fna.gz
  DOWNLOAD_SUCCESS = True
elif CASO == "caso_C":
  print("── CASO C: Streptomyces venezuelae (~8.2 Mb, GC ~72%) ───")
  !wget -q ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/006/SRR2589046/SRR2589046_1.fastq.gz -O {CASO}/data/SRR2589046_1.fastq.gz
  !wget -q ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/006/SRR2589046/SRR2589046_2.fastq.gz -O {CASO}/data/SRR2589046_2.fastq.gz
  !wget -q "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/253/235/GCF_000253235.1_ASM25323v1/GCF_000253235.1_ASM25323v1_genomic.fna.gz" -O {CASO}/data/ref.fna.gz
  DOWNLOAD_SUCCESS = True
else:
  print("⚠️ Error: CASO no reconocido. Por favor, define 'CASO' como 'caso_A', 'caso_B' o 'caso_C'.")

# Descomprimir el genoma de referencia y mostrar mensaje de éxito si la descarga fue exitosa
if DOWNLOAD_SUCCESS:
  print(f"Descomprimiendo {gz_file_path}...")
  !gunzip {CASO}/data/ref.fna.gz
  print(f"✅ Caso {CASO} listo")

## 👀 Paso 1.1 — Inspeccionar los archivos descargados

Vamos a verificar que los archivos FASTQ y el genoma de referencia FASTA se hayan descargado correctamente y tengan el formato esperado. Revisaremos el encabezado de cada uno.

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

print(f"=== Encabezado del archivo FASTQ 1 para {CASO} ===")
!zcat {CASO}/data/*_1.fastq.gz | head -n 8

print(f"\n=== Encabezado del archivo FASTQ 2 para {CASO} ===")
!zcat {CASO}/data/*_2.fastq.gz | head -n 8

print(f"\n=== Encabezado del archivo FASTA de referencia para {CASO} ===")
!head -n 5 {CASO}/data/ref.fna

## 🔎 Paso 2 — Control de calidad y limpieza con fastp

fastp realiza en un solo paso: detección y recorte de adaptadores, filtrado por calidad y generación de reporte.
Ajuste `CASO` al directorio de su caso (caso_A, caso_B o caso_C).

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

!mkdir -p {CASO}/results/qc_results

!fastp \
  --in1  {CASO}/data/*_1.fastq.gz \
  --in2  {CASO}/data/*_2.fastq.gz \
  --out1 {CASO}/results/clean_R1.fastq.gz \
  --out2 {CASO}/results/clean_R2.fastq.gz \
  --html {CASO}/results/qc_results/fastp_report.html \
  --json {CASO}/results/qc_results/fastp_report.json \
  --qualified_quality_phred 30 \
  --length_required 50 \
  --thread 4

!echo "✅ Limpieza completa. Reporte en {CASO}/results/qc_results/fastp_report.html"

## 📊 Paso 3 — Explorar el reporte de fastp con Python


In [ ]:
import json, pathlib
import matplotlib.pyplot as plt

report = json.loads(pathlib.Path(f"{CASO}/results/qc_results/fastp_report.json").read_text())

summary = report["summary"]
before  = summary["before_filtering"]
after   = summary["after_filtering"]

print("=== Resumen de calidad ===")
print(f"{'':30s} {'Antes':>12} {'Después':>12}")
print("-" * 56)
print(f"{'Lecturas totales':30s} {before['total_reads']:>12,} {after['total_reads']:>12,}")
print(f"{'Bases totales':30s} {before['total_bases']:>12,} {after['total_bases']:>12,}")
print(f"{'Calidad Q20 (%':30s} {before['q20_rate']*100:>11.1f}% {after['q20_rate']*100:>11.1f}%")
print(f"{'Calidad Q30 (%':30s} {before['q30_rate']*100:>11.1f}% {after['q30_rate']*100:>11.1f}%")
print(f"{'GC (%':30s} {before['gc_content']*100:>11.1f}% {after['gc_content']*100:>11.1f}%")

### Comparación de Métricas de Calidad (Antes vs. Después de `fastp`)

Este gráfico compara visualmente las métricas clave del resumen de `fastp` (número de lecturas, bases, y tasas de calidad Q20, Q30 y GC-content) antes y después del proceso de filtrado. Esto nos ayuda a entender el impacto de `fastp` en la calidad de nuestros datos.

In [ ]:
import matplotlib.pyplot as plt
import os

# Extraer los diccionarios de curvas de calidad
quality_curves_r1_before = report['read1_before_filtering']['quality_curves']
quality_curves_r1_after = report['read1_after_filtering']['quality_curves']
quality_curves_r2_before = report['read2_before_filtering']['quality_curves']
quality_curves_r2_after = report['read2_after_filtering']['quality_curves']

# Función para calcular la media de calidad por ciclo a partir del diccionario de curvas
def calculate_mean_q_per_cycle(quality_curves_dict):
    # Verificar si el diccionario está vacío o si las listas de bases están vacías
    if not quality_curves_dict or not quality_curves_dict.get('A'):
        return [], [] # Retornar listas vacías si no hay datos

    num_cycles = len(quality_curves_dict['A']) # Asumiendo que todas las listas tienen la misma longitud
    cumulative_cycles = list(range(1, num_cycles + 1))
    mean_q_per_cycle = []
    for i in range(num_cycles):
        # Promediar las calidades de A, C, G, T para cada ciclo
        cycle_qualities = [
            quality_curves_dict['A'][i],
            quality_curves_dict['C'][i],
            quality_curves_dict['G'][i],
            quality_curves_dict['T'][i]
        ]
        mean_q_per_cycle.append(sum(cycle_qualities) / len(cycle_qualities))
    return cumulative_cycles, mean_q_per_cycle

# Preparar datos para el plot para Read 1 antes de fastp
cumulative_cycles_r1_before, mean_q_r1_before = calculate_mean_q_per_cycle(quality_curves_r1_before)

# Preparar datos para el plot para Read 1 después de fastp
cumulative_cycles_r1_after, mean_q_r1_after = calculate_mean_q_per_cycle(quality_curves_r1_after)

# Preparar datos para el plot para Read 2 antes de fastp
cumulative_cycles_r2_before, mean_q_r2_before = calculate_mean_q_per_cycle(quality_curves_r2_before)

# Preparar datos para el plot para Read 2 después de fastp
cumulative_cycles_r2_after, mean_q_r2_after = calculate_mean_q_per_cycle(quality_curves_r2_after)

# Crear los gráficos de calidad por base
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

# Rango de calidad para las zonas de color
red_zone_max = 20
yellow_zone_max = 30

# Gráfico para Read 1
if cumulative_cycles_r1_before and mean_q_r1_before:
    # Zonas de color
    axes[0].axhspan(0, red_zone_max, color='red', alpha=0.1, label='Calidad Baja (Q<20)')
    axes[0].axhspan(red_zone_max, yellow_zone_max, color='yellow', alpha=0.1, label='Calidad Media (20<=Q<30)')
    axes[0].axhspan(yellow_zone_max, 40, color='green', alpha=0.1, label='Calidad Alta (Q>=30)')

    axes[0].plot(cumulative_cycles_r1_before, mean_q_r1_before, label='Antes de fastp', color='red', alpha=0.7)
    axes[0].plot(cumulative_cycles_r1_after, mean_q_r1_after, label='Después de fastp', color='green', alpha=0.7)
    axes[0].set_title(f'Calidad Media por Base - Read 1 para {CASO}')
    axes[0].set_xlabel('Posición de Base (Ciclo)')
    axes[0].set_ylabel('Calidad Media (Phred Score)')
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)
    axes[0].legend()
    axes[0].set_ylim(0, 40) # Rango típico de Phred score
else:
    axes[0].text(0.5, 0.5, 'No hay datos para Read 1', horizontalalignment='center', verticalalignment='center', transform=axes[0].transAxes, fontsize=12, color='gray')
    axes[0].set_title(f'Calidad Media por Base - Read 1 para {CASO}')
    axes[0].set_xlabel('Posición de Base (Ciclo)')
    axes[0].set_ylabel('Calidad Media (Phred Score)')

# Gráfico para Read 2
if cumulative_cycles_r2_before and mean_q_r2_before:
    # Zonas de color
    axes[1].axhspan(0, red_zone_max, color='red', alpha=0.1, label='Calidad Baja (Q<20)')
    axes[1].axhspan(red_zone_max, yellow_zone_max, color='yellow', alpha=0.1, label='Calidad Media (20<=Q<30)')
    axes[1].axhspan(yellow_zone_max, 40, color='green', alpha=0.1, label='Calidad Alta (Q>=30)')

    axes[1].plot(cumulative_cycles_r2_before, mean_q_r2_before, label='Antes de fastp', color='red', alpha=0.7)
    axes[1].plot(cumulative_cycles_r2_after, mean_q_r2_after, label='Después de fastp', color='green', alpha=0.7)
    axes[1].set_title(f'Calidad Media por Base - Read 2 para {CASO}')
    axes[1].set_xlabel('Posición de Base (Ciclo)')
    axes[1].set_ylabel('Calidad Media (Phred Score)')
    axes[1].grid(axis='y', linestyle='--', alpha=0.7)
    axes[1].legend()
    axes[1].set_ylim(0, 40) # Rango típico de Phred score
else:
    axes[1].text(0.5, 0.5, 'No hay datos para Read 2', horizontalalignment='center', verticalalignment='center', transform=axes[1].transAxes, fontsize=12, color='gray')
    axes[1].set_title(f'Calidad Media por Base - Read 2 para {CASO}')
    axes[1].set_xlabel('Posición de Base (Ciclo)')
    axes[1].set_ylabel('Calidad Media (Phred Score)')

plt.tight_layout()
plt.show()

# Guardar el gráfico en la carpeta qc_results
output_dir = f"{CASO}/results/qc_results"
os.makedirs(output_dir, exist_ok=True)
plt.savefig(os.path.join(output_dir, f"quality_plot_{CASO}.png"))
print(f"✅ Gráfico de calidad guardado en {output_dir}/quality_plot_{CASO}.png")

## 🔧 Paso 4 — Ensamblaje de novo con SPAdes

SPAdes (St. Petersburg Assembler) es un software de ensamblaje de novo que se utiliza para reconstruir genomas a partir de lecturas de secuencias de ADN. Está diseñado para trabajar con lecturas de secuenciación de alto rendimiento (como Illumina) y puede manejar diferentes tipos de datos, incluyendo lecturas emparejadas (paired-end) y lecturas de longitud variable. SPAdes es conocido por su precisión y capacidad para ensamblar genomas complejos, y es una herramienta ampliamente utilizada en bioinformática para proyectos de secuenciación de genomas bacterianos y eucariotas pequeños.

NOTA: Este paso demora entre 30-50 minutos

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

!spades.py \
  -1 {CASO}/results/clean_R1.fastq.gz \
  -2 {CASO}/results/clean_R2.fastq.gz \
  -o {CASO}/results/spades_assembly \
  --careful \
  -t 4 \
  -m 12

!echo "✅ Ensamblaje completo. Contigs en {CASO}/results/spades_assembly/contigs.fasta"

## 📏 Paso 5 — Evaluación del ensamblaje con QUAST

QUAST compara los contigs obtenidos contra el genoma de referencia para calcular métricas de calidad.
Ver la definición de N50, L50 y Genome Fraction en las secciones **6** y **7.5** del [README del Módulo 5](../README.md).

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

!quast.py \
  {CASO}/results/spades_assembly/contigs.fasta \
  -r {CASO}/data/ref.fna \
  -o {CASO}/results/quast_results \
  --threads 4

!echo "✅ QUAST completo. Reporte en {CASO}/results/quast_results/report.html"

## 📊 Paso 6 — Leer y visualizar los resultados de QUAST con Python

In [ ]:
import pandas as pd, pathlib

report_path = pathlib.Path(f"{CASO}/results/quast_results/report.tsv")

df = pd.read_csv(report_path, sep="\t", header=0, index_col=0)
print(df.to_string())

## ❓ Preguntas para reflexionar

Responda estas preguntas en una celda de texto nueva (`+ Texto`) al final del notebook:

1. ¿Cuántos contigs produjo SPAdes? ¿Cómo se compara con lo esperado para una bacteria de este tamaño?
2. ¿Cuál es el N50 y el L50 del ensamblaje? ¿Qué indican?
3. ¿Qué porcentaje del genoma de referencia fue cubierto (*Genome Fraction*)?
4. ¿Cuántos mismatches e indels por 100 kbp reporta QUAST?
5. Compare la calidad Q30 antes y después del trimming con fastp. ¿Mejoró significativamente?
6. **Caso C solamente:** ¿Cómo se refleja el alto contenido GC (~72%) en los resultados de fastp y en el ensamblaje?

---
*Para repasar los conceptos de N50, L50, Genome Fraction y métricas de evaluación, consulte las secciones 6 y 7.5 del [README del Módulo 5](../README.md).*